# Vector Graph Memory Playground

This notebook demonstrates the basic setup of a PydanticAI agent with Qdrant and JanusGraph database tools.

## Setup and Imports

In [ ]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from gremlin_python.driver import client as gremlin_client
from pydantic_ai import Agent, RunContext
from pydantic import BaseModel

from vector_graph_memory.embeddings import OpenAIEmbeddingProvider, SentenceTransformerEmbeddingProvider
from vector_graph_memory.memory import VectorGraphMemory

# Load environment variables
load_dotenv()

## Database Connections

In [ ]:
# Qdrant connection
qdrant_host = os.getenv('QDRANT_HOST', 'localhost')
qdrant_port = int(os.getenv('QDRANT_PORT', '6333'))

qdrant = QdrantClient(host=qdrant_host, port=qdrant_port)
print(f"Connected to Qdrant at {qdrant_host}:{qdrant_port}")

In [ ]:
# JanusGraph connection
janusgraph_host = os.getenv('JANUSGRAPH_HOST', 'localhost')
janusgraph_port = int(os.getenv('JANUSGRAPH_PORT', '8182'))

janus = gremlin_client.Client(
    f'ws://{janusgraph_host}:{janusgraph_port}/gremlin',
    'g'
)
print(f"Connected to JanusGraph at {janusgraph_host}:{janusgraph_port}")

# Initialize embedding provider (configurable)
embedding_provider_type = os.getenv('EMBEDDING_PROVIDER', 'openai')

if embedding_provider_type == 'openai':
    embedding_provider = OpenAIEmbeddingProvider(
        model=os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')
    )
elif embedding_provider_type == 'sentence-transformers':
    embedding_provider = SentenceTransformerEmbeddingProvider(
        model_name=os.getenv('SENTENCE_TRANSFORMER_MODEL', 'all-MiniLM-L6-v2')
    )
else:
    raise ValueError(f"Unknown embedding provider: {embedding_provider_type}")

print(f"Initialized {embedding_provider_type} embedding provider (dim={embedding_provider.dimension})")

## Database Context

In [ ]:
class DatabaseContext(BaseModel):
    """Context containing memory system for the agent."""
    memory: VectorGraphMemory
    
    class Config:
        arbitrary_types_allowed = True

## Agent Tools

In [ ]:
# Create the agent with configurable LLM model
llm_model = os.getenv('LLM_MODEL', 'openai:gpt-4')

agent = Agent(
    llm_model,
    deps_type=DatabaseContext,
    system_prompt="You are a helpful assistant with access to a vector-graph memory system."
)

@agent.tool
async def search_vector_memory(ctx: RunContext[DatabaseContext], query: str, limit: int = 5) -> str:
    """
    Search the vector database for semantically similar content.
    
    Args:
        query: The search query text
        limit: Maximum number of results to return
    
    Returns:
        Search results as a formatted string
    """
    results = await ctx.deps.memory.search_similar(query, limit=limit)
    
    if not results:
        return f"No results found for query: '{query}'"
    
    formatted_results = []
    for i, result in enumerate(results, 1):
        formatted_results.append(
            f"{i}. [{result['type']}] {result['content']} (score: {result['score']:.3f})"
        )
    
    return "\\n".join(formatted_results)

@agent.tool
async def query_graph(ctx: RunContext[DatabaseContext], start_id: str, traversal_steps: str) -> str:
    """
    Execute a graph traversal starting from a specific entity.
    
    Args:
        start_id: Starting entity ID
        traversal_steps: Gremlin traversal steps (e.g., "out('works_at').values('content')")
    
    Returns:
        Query results as a formatted string
    """
    try:
        results = await ctx.deps.memory.query_graph_traversal(start_id, traversal_steps)
        return f"Graph traversal results: {results}"
    except Exception as e:
        return f"Error executing graph traversal: {str(e)}"

@agent.tool
async def add_to_memory(ctx: RunContext[DatabaseContext], content: str, entity_type: str, metadata: dict | None = None) -> str:
    """
    Add new content to both vector and graph databases.
    
    Args:
        content: The content to store
        entity_type: The type of entity (e.g., 'job', 'company', 'person')
        metadata: Optional additional metadata
    
    Returns:
        Confirmation message with entity ID
    """
    entity_id = await ctx.deps.memory.add_entity(content, entity_type, metadata)
    return f"Added {entity_type} to memory with ID: {entity_id}"

@agent.tool
async def link_entities(ctx: RunContext[DatabaseContext], from_id: str, to_id: str, relationship: str) -> str:
    """
    Create a relationship between two entities in the graph.
    
    Args:
        from_id: Source entity ID
        to_id: Target entity ID
        relationship: Relationship type (e.g., 'works_at', 'applied_to')
    
    Returns:
        Confirmation message
    """
    try:
        await ctx.deps.memory.add_relationship(from_id, to_id, relationship)
        return f"Created '{relationship}' relationship from {from_id} to {to_id}"
    except Exception as e:
        return f"Error creating relationship: {str(e)}"

print(f"Agent initialized with LLM: {llm_model}")
print("Agent tools registered:")
print("- search_vector_memory: Search for semantically similar content")
print("- query_graph: Execute graph traversals from specific entities")
print("- add_to_memory: Store new entities in the memory system")
print("- link_entities: Create relationships between entities")

## Test the Agent

In [ ]:
# Initialize the memory system
memory = VectorGraphMemory(
    qdrant_client=qdrant,
    janus_client=janus,
    embedding_provider=embedding_provider,
    collection_name="job_search_memory"
)

# Create database context
db_context = DatabaseContext(memory=memory)

# Run a simple query
result = await agent.run(
    "What tools do you have available?",
    deps=db_context
)

print(result.data)

## Example: Job Search Use Case

In [ ]:
# Example: Add a job to memory
result = await agent.run(
    "Add a software engineer position at Google to my memory",
    deps=db_context
)

print(result.data)

In [ ]:
# Example: Search for jobs
result = await agent.run(
    "Search for engineering positions in my memory",
    deps=db_context
)

print(result.data)

## Cleanup

In [ ]:
# Close connections
janus.close()
print("Database connections closed")